# Meet weather

Adds three weather columns to `regular_season.csv` and `bigwest_2019_2026_results.csv`, describing where and when each meet was held:

| Column | Meaning |
|---|---|
| `temp` | Day's high temperature (°F), averaged over the meet's days |
| `precip_in` | Day's total precipitation (inches, rain plus any snow as water), averaged over the meet's days |
| `humidity_max` | Day's highest relative humidity (%), averaged over the meet's days |

Each one is averaged over the meet's days (a 2-day meet gets the mean of its two days) because we don't know which day each athlete competed.

**How:**
1. **Where each meet was held.** Regular-season meets: each athlete's TFRRS page links to every meet they ran, and each meet's TFRRS page lists its venue and city (e.g. "UC Irvine - Irvine, CA"). Championships: the same location line on the championship's TFRRS page. Locations already saved in `meet_weather.csv` are reused; only meets that aren't there yet (e.g. the 2019 season) are looked up on TFRRS.
2. **City → coordinates** with the [Open-Meteo geocoding API](https://open-meteo.com/en/docs/geocoding-api).
3. **Weather** from the [Open-Meteo historical weather API](https://open-meteo.com/en/docs/historical-weather-api), a free weather-history service built from weather-model data, not a station at the track. Model precipitation is approximate: it can show a little drizzle on dry days and miss small local showers.

**Championship dates:** TFRRS lists 2026 as May 8–16 because it folds the multi-events weekend (decathlon/heptathlon, May 8–9 in Irvine) into the main meet (May 15–16 in Long Beach). `CHAMP_DATE_OVERRIDES` uses the main-meet days, so the weather describes the meet almost every athlete competed in.

**Indoor meets get no weather** (left blank): the outdoor weather doesn't describe conditions inside. Indoor meets are flagged with the same rule as the predictor notebook.

`meet_weather.csv` keeps one row per meet with its TFRRS link, location, coordinates and weather, so every value can be checked.

In [1]:
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

DATA_DIR = Path('../data/processed')
REGULAR_SEASON_PATH = DATA_DIR / 'regular_season.csv'
RESULTS_PATH = DATA_DIR / 'bigwest_2019_2026_results.csv'
MEET_WEATHER_PATH = DATA_DIR / 'meet_weather.csv'  # one row per meet: location + weather

WEATHER_COLS = ['temp', 'precip_in', 'humidity_max']

REFRESH_LOCATIONS = False  # True = look up every meet on TFRRS again, even ones saved in meet_weather.csv
CACHE_DIR = None           # optional folder to keep downloaded TFRRS pages between runs
PAUSE_SECONDS = 1.0        # wait between TFRRS requests, to go easy on the site

# Main-meet days where the TFRRS header also covers a separate multi-events weekend
# (2026: multis May 8-9 in Irvine, main meet May 15-16 in Long Beach; bigwest.org)
CHAMP_DATE_OVERRIDES = {2026: 'May 15-16, 2026'}

# Read every column as text so the CSVs are written back exactly as they were, plus the weather columns
regular_season = pd.read_csv(REGULAR_SEASON_PATH, dtype=str, keep_default_na=False)
results = pd.read_csv(RESULTS_PATH, dtype=str, keep_default_na=False)
regular_season_columns, results_columns = list(regular_season.columns), list(results.columns)
regular_season = regular_season.drop(columns=WEATHER_COLS, errors='ignore')
results = results.drop(columns=WEATHER_COLS, errors='ignore')
saved = pd.read_csv(MEET_WEATHER_PATH) if MEET_WEATHER_PATH.exists() and not REFRESH_LOCATIONS else None
print(f'{len(regular_season)} regular-season rows, {len(results)} championship rows')

57300 regular-season rows, 5257 championship rows


## Helpers

In [2]:
SESSION = requests.Session()
SESSION.headers['User-Agent'] = 'Mozilla/5.0'


def norm(text):
    return re.sub(r'\s+', ' ', str(text)).strip()


def fetch(url, cache_name):
    """GET a TFRRS page, politely, with an optional on-disk cache."""
    cache_path = Path(CACHE_DIR) / cache_name if CACHE_DIR else None
    if cache_path and cache_path.exists():
        return cache_path.read_text(encoding='utf-8')
    for attempt in range(3):
        resp = SESSION.get(url, timeout=30)
        if resp.status_code == 200:
            if cache_path:
                cache_path.parent.mkdir(parents=True, exist_ok=True)
                cache_path.write_text(resp.text, encoding='utf-8')
            time.sleep(PAUSE_SECONDS)
            return resp.text
        time.sleep(5 * (attempt + 1))
    raise RuntimeError(f'HTTP {resp.status_code} for {url}')


MEET_LINK = re.compile(r'/results/(?:xc/)?(\d+)/[^/#?]+/?$')


def athlete_meets(html):
    """(meet_name, meet_date, tfrrs_meet_id, url) for each meet listed on a TFRRS athlete page."""
    soup = BeautifulSoup(html, 'html.parser')
    rows = []
    for a in soup.select('a[href*="/results/"]'):
        m = MEET_LINK.search(a['href'])
        if not m:
            continue  # links to a single event, not the meet
        name = norm(a.get_text())
        container = a.find_parent(['th', 'tr', 'div'])
        date = norm(norm(container.get_text(' ')).replace(name, '', 1)) if container else ''
        rows.append((name, date, int(m.group(1)), a['href']))
    return rows


def meet_header(html):
    """Date line and 'Venue - City, ST' line from the header of a TFRRS meet page."""
    soup = BeautifulSoup(html, 'html.parser')
    parts = [norm(el.get_text(' ')) for el in soup.select('.panel-heading-normal-text')]
    parts = [p for p in parts if p and p != '|']
    return {'tfrrs_date': parts[0] if parts else None,
            'location': parts[1] if len(parts) > 1 else None}


def split_location(text):
    """'UC Irvine - Irvine, CA' -> ('UC Irvine', 'Irvine', 'CA')."""
    if not isinstance(text, str):
        return None, None, None
    venue, _, place = text.rpartition(' - ')
    m = re.match(r'^(.+?),\s*([A-Z]{2})$', place.strip())
    if not m:
        return venue or None, None, None
    return venue or None, m.group(1).strip(), m.group(2)


DATE = re.compile(r'^([A-Z][a-z]{2})\s+(\d{1,2})(?:\s*-\s*(?:([A-Z][a-z]{2})\s+)?(\d{1,2}))?,\s*(\d{4})$')


def date_range(text):
    """'Mar 31-Apr 1, 2023' -> (2023-03-31, 2023-04-01)."""
    m = DATE.match(text.strip())
    if not m:
        raise ValueError(f'Unrecognized meet date: {text!r}')
    mon1, day1, mon2, day2, year = m.groups()
    return pd.Timestamp(f'{mon1} {day1} {year}'), pd.Timestamp(f'{mon2 or mon1} {day2 or day1} {year}')


STATES = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California',
    'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'DC': 'District of Columbia',
    'FL': 'Florida', 'GA': 'Georgia', 'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois',
    'IN': 'Indiana', 'IA': 'Iowa', 'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana',
    'ME': 'Maine', 'MD': 'Maryland', 'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota',
    'MS': 'Mississippi', 'MO': 'Missouri', 'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada',
    'NH': 'New Hampshire', 'NJ': 'New Jersey', 'NM': 'New Mexico', 'NY': 'New York',
    'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio', 'OK': 'Oklahoma', 'OR': 'Oregon',
    'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina', 'SD': 'South Dakota',
    'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont', 'VA': 'Virginia',
    'WA': 'Washington', 'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming',
}


def geocode(city, state):
    """(latitude, longitude) of a US city, via Open-Meteo's geocoding API."""
    resp = requests.get('https://geocoding-api.open-meteo.com/v1/search',
                        params={'name': city, 'count': 20, 'country_code': 'US'}, timeout=30)
    resp.raise_for_status()
    for r in resp.json().get('results', []):
        if r.get('admin1') == STATES.get(state):
            return r['latitude'], r['longitude']
    return np.nan, np.nan


def daily_weather(lat, lon, start, end):
    """Daily high temperature (°F), precipitation (inches) and peak humidity (%) from Open-Meteo's historical API."""
    for attempt in range(5):
        resp = requests.get('https://archive-api.open-meteo.com/v1/archive', params={
            'latitude': lat, 'longitude': lon,
            'start_date': start.strftime('%Y-%m-%d'), 'end_date': end.strftime('%Y-%m-%d'),
            'daily': 'temperature_2m_max,precipitation_sum,relative_humidity_2m_max',
            'temperature_unit': 'fahrenheit', 'precipitation_unit': 'inch', 'timezone': 'auto',
        }, timeout=30)
        if resp.status_code != 429:  # 429 = rate limited: wait and retry
            break
        time.sleep(30 * (attempt + 1))
    resp.raise_for_status()
    daily = resp.json()['daily']
    return pd.DataFrame({'temp': daily['temperature_2m_max'],
                         'precip_in': daily['precipitation_sum'],
                         'humidity_max': daily['relative_humidity_2m_max']}, dtype=float)


## 1. Where each meet was held

**Regular season:** meets already in `meet_weather.csv` reuse their saved location. For the rest, pick the fewest athletes whose TFRRS pages together list every missing meet, read each meet's TFRRS link from those pages, then read the location from each meet's page.

In [3]:
meets = regular_season[['meet_name', 'meet_date']].drop_duplicates().reset_index(drop=True)
LINK_COLS = ['tfrrs_meet_id', 'tfrrs_url', 'location']

if saved is not None:
    season_saved = saved.loc[saved['source'] == 'regular_season', ['meet_name', 'meet_date'] + LINK_COLS]
    season_meets = meets.merge(season_saved, on=['meet_name', 'meet_date'], how='left')
else:
    season_meets = meets.assign(**{c: np.nan for c in LINK_COLS})
season_meets['tfrrs_url'] = season_meets['tfrrs_url'].astype(object)
season_meets['location'] = season_meets['location'].astype(object)
todo = season_meets[season_meets['location'].isna()]
print(f'Reused saved locations for {len(meets) - len(todo)} of {len(meets)} meets; looking up {len(todo)} on TFRRS')

if len(todo):
    # Fewest athletes whose pages cover every missing meet (greedy set cover)
    meet_keys = pd.Series(list(zip(regular_season['meet_name'], regular_season['meet_date'])))
    meets_by_athlete = meet_keys.groupby(regular_season['athlete_id'].astype(int).values).apply(set).to_dict()
    uncovered = set(zip(todo['meet_name'], todo['meet_date']))
    athletes = []
    while uncovered:
        best = max(meets_by_athlete, key=lambda a: len(meets_by_athlete[a] & uncovered))
        athletes.append(best)
        uncovered -= meets_by_athlete[best]
    print(f'Reading {len(athletes)} athlete pages to find {len(todo)} meets')

    links = []
    for athlete_id in tqdm(athletes, desc='Athlete pages'):
        links += athlete_meets(fetch(f'https://www.tfrrs.org/athletes/{athlete_id}.html',
                                     f'athletes/{athlete_id}.html'))
    links = (pd.DataFrame(links, columns=['meet_name', 'date_norm', 'tfrrs_meet_id', 'tfrrs_url'])
             .drop_duplicates(['meet_name', 'date_norm', 'tfrrs_meet_id']))
    found = (todo[['meet_name', 'meet_date']].assign(date_norm=todo['meet_date'].map(norm))
             .merge(links, on=['meet_name', 'date_norm'], how='inner').drop(columns='date_norm'))
    assert not found.duplicated(['meet_name', 'meet_date']).any(), 'A meet matched two TFRRS pages'

    pages = found.drop_duplicates('tfrrs_meet_id')
    print(f'Reading {len(pages)} meet pages')
    headers = {row.tfrrs_meet_id: meet_header(fetch(row.tfrrs_url, f'meets/{int(row.tfrrs_meet_id)}.html'))
               for row in tqdm(pages.itertuples(), total=len(pages), desc='Meet pages')}
    found['location'] = found['tfrrs_meet_id'].map(lambda m: headers.get(m, {}).get('location'))

    season_meets = season_meets.set_index(['meet_name', 'meet_date'])
    season_meets.update(found.set_index(['meet_name', 'meet_date'])[LINK_COLS])
    season_meets = season_meets.reset_index()

missing = season_meets[season_meets['location'].isna()]
print(f'{len(season_meets) - len(missing)} of {len(season_meets)} meets have a location')
if len(missing):
    display(missing)

Reused saved locations for 372 of 437 meets; looking up 65 on TFRRS
Reading 20 athlete pages to find 65 meets


Athlete pages:   0%|          | 0/20 [00:00<?, ?it/s]

Reading 65 meet pages


Meet pages:   0%|          | 0/65 [00:00<?, ?it/s]

437 of 437 meets have a location


**Championships:** saved years reuse their location from `meet_weather.csv`. For a new year, read a few of that year's championship athletes' TFRRS pages, find the Big West championship link with that year's date, and read the date and location from the meet page.

In [4]:
def find_championship(year):
    """Date, location and TFRRS link of the Big West outdoor championships in `year`, via athletes' TFRRS pages."""
    athlete_ids = results.loc[results['meet_year'] == year, 'athlete_id'].str.replace(r'\.0$', '', regex=True)
    for athlete_id in athlete_ids.drop_duplicates().head(10):
        for name, date, meet_id, url in athlete_meets(fetch(f'https://www.tfrrs.org/athletes/{athlete_id}.html',
                                                            f'athletes/{athlete_id}.html')):
            if ('big west' in name.lower() and 'indoor' not in name.lower()
                    and date.endswith(year) and date[:3] in ('Apr', 'May')):
                header = meet_header(fetch(url, f'meets/{meet_id}.html'))
                return {'tfrrs_meet_id': meet_id, 'tfrrs_url': url, **header}
    raise LookupError(f'No Big West championship link found for {year}')


saved_champs = (saved.loc[saved['source'] == 'championship'].assign(meet_year=lambda d: d['meet_year'].astype(int).astype(str))
                .set_index('meet_year') if saved is not None else pd.DataFrame())
champ_rows = []
for year in tqdm(sorted(results['meet_year'].unique()), desc='Championships'):
    if year in saved_champs.index:
        row = saved_champs.loc[year]
        champ_rows.append({'meet_year': year, 'meet_date': row['meet_date'], 'location': row['location'],
                           'tfrrs_meet_id': row['tfrrs_meet_id'], 'tfrrs_url': row['tfrrs_url']})
        continue
    found = find_championship(year)
    meet_date = CHAMP_DATE_OVERRIDES.get(int(year), found['tfrrs_date'])
    if meet_date != found['tfrrs_date']:
        print(f"{year}: TFRRS lists {found['tfrrs_date']}; using the main meet, {meet_date}")
    champ_rows.append({'meet_year': year, 'meet_date': meet_date, 'location': found['location'],
                       'tfrrs_meet_id': found['tfrrs_meet_id'], 'tfrrs_url': found['tfrrs_url']})
champ_meets = pd.DataFrame(champ_rows)
champ_meets

Championships:   0%|          | 0/7 [00:00<?, ?it/s]

,meet_year,meet_date,location,tfrrs_meet_id,tfrrs_url
0,2019,"May 10-11, 2019","UC Santa Barbara (Outdoor) - Santa Barbara, CA",62142.0,https://www.tfrrs.org/results/62142/Big_West_O...
1,2021,"May 12-15, 2021","UC Irvine - Irvine, CA",NaN,NaN
2,2022,"May 13-14, 2022","UC Davis - Davis, CA",NaN,NaN
3,2023,"May 12-13, 2023","Cal St. Fullerton - Fullerton, CA",NaN,NaN
4,2024,"May 10-11, 2024","Cal Poly-SLO - San Luis Obispo, CA",NaN,NaN
5,2025,"May 16-17, 2025","Jack Rose Track - Long Beach, CA",NaN,NaN
6,2026,"May 15-16, 2026","Jack Rose Track - Long Beach, CA",NaN,NaN


## 2. Indoor meets

Same rule as the predictor notebook: "Indoor" in the name, or no outdoor-only events and either indoor-only events or a January/February date.

In [5]:
INDOOR_ONLY_EVENTS = {'60', '60H', 'WT', 'Pent'}
OUTDOOR_ONLY_EVENTS = {'100', '100H', '110H', '400H', '3000S', '10,000', 'DT', 'HT', 'JT', '4x100', 'Dec'}

events = regular_season.groupby(['meet_name', 'meet_date'])['event'].agg(set)
start_month = regular_season.groupby(['meet_name', 'meet_date'])['meet_start_date'].first().str[5:7].astype(int)
named_indoor = pd.Series(events.index.get_level_values('meet_name').str.contains('indoor', case=False),
                         index=events.index)
has_outdoor = events.map(lambda e: bool(e & OUTDOOR_ONLY_EVENTS))
has_indoor = events.map(lambda e: bool(e & INDOOR_ONLY_EVENTS))
is_indoor = (named_indoor | (~has_outdoor & (has_indoor | (start_month <= 2)))).rename('is_indoor')

season_meets = season_meets.drop(columns='is_indoor', errors='ignore').merge(
    is_indoor.reset_index(), on=['meet_name', 'meet_date'], how='left')
print(season_meets['is_indoor'].value_counts().rename({True: 'indoor', False: 'outdoor'}).to_string())

is_indoor
outdoor    316
indoor     121


## 3. Coordinates and weather

In [6]:
season_meets['source'] = 'regular_season'
champ_meets['source'] = 'championship'
champ_meets['meet_name'] = 'Big West Championships ' + champ_meets['meet_year']
champ_meets['is_indoor'] = False
meet_weather = pd.concat([season_meets, champ_meets], ignore_index=True)

meet_weather[['venue', 'city', 'state']] = meet_weather['location'].apply(lambda t: pd.Series(split_location(t)))
dates = meet_weather['meet_date'].map(date_range)
meet_weather['start_date'] = [start for start, _ in dates]
meet_weather['end_date'] = [end for _, end in dates]

places = meet_weather[['city', 'state']].dropna().drop_duplicates()
coords = {(c, s): geocode(c, s) for c, s in tqdm(places.itertuples(index=False), total=len(places), desc='Geocoding')}
meet_weather['latitude'] = [coords.get((c, s), (np.nan, np.nan))[0] for c, s in zip(meet_weather['city'], meet_weather['state'])]
meet_weather['longitude'] = [coords.get((c, s), (np.nan, np.nan))[1] for c, s in zip(meet_weather['city'], meet_weather['state'])]
print(f'Geocoded {sum(pd.notna(v[0]) for v in coords.values())} of {len(coords)} cities')

def meet_conditions(row):
    """Each weather column averaged over the meet's days; blank for indoor meets and unknown locations."""
    if row.is_indoor or pd.isna(row.latitude):
        return {c: np.nan for c in WEATHER_COLS}
    days = daily_weather(row.latitude, row.longitude, row.start_date, row.end_date)
    return {'temp': round(days['temp'].mean(), 1),
            'precip_in': round(days['precip_in'].mean(), 2),
            'humidity_max': round(days['humidity_max'].mean(), 1)}

conditions = [meet_conditions(row) for row in tqdm(meet_weather.itertuples(), total=len(meet_weather), desc='Weather')]
meet_weather[WEATHER_COLS] = pd.DataFrame(conditions, index=meet_weather.index)

cols = ['source', 'meet_name', 'meet_year', 'meet_date', 'start_date', 'end_date', 'tfrrs_meet_id',
        'tfrrs_url', 'location', 'venue', 'city', 'state', 'latitude', 'longitude', 'is_indoor'] + WEATHER_COLS
meet_weather = meet_weather[cols]
meet_weather.to_csv(MEET_WEATHER_PATH, index=False)
for c in WEATHER_COLS:
    print(f'\n{c}:')
    print(meet_weather.groupby('source')[c].describe().round(2).to_string())

Geocoding:   0%|          | 0/60 [00:00<?, ?it/s]

Geocoded 60 of 60 cities


Weather:   0%|          | 0/444 [00:00<?, ?it/s]


temp:
                count   mean   std   min    25%    50%    75%   max
source                                                             
championship      7.0  73.19  8.26  65.2  68.20  70.10  75.40  89.8
regular_season  316.0  70.71  9.98  50.2  63.02  69.65  77.93  98.5

precip_in:
                count  mean   std  min  25%  50%   75%   max
source                                                      
championship      7.0  0.01  0.02  0.0  0.0  0.0  0.02  0.04
regular_season  316.0  0.05  0.16  0.0  0.0  0.0  0.02  1.84

humidity_max:
                count   mean    std   min   25%    50%   75%    max
source                                                             
championship      7.0  91.31   4.17  87.5  88.5  89.20  93.5   98.5
regular_season  316.0  86.28  14.90  29.0  81.5  91.75  96.5  100.0


## 4. Add the weather columns to the CSVs

In [7]:
FORMATS = {'temp': '{:.1f}', 'precip_in': '{:.2f}', 'humidity_max': '{:.1f}'}

def as_text(df):
    """Weather columns as text (blank when missing), matching the all-text CSVs."""
    return df.assign(**{c: df[c].map(lambda v, f=FORMATS[c]: '' if pd.isna(v) else f.format(v)) for c in WEATHER_COLS})

def restore_order(df, original_columns):
    """Keep columns where they were on a rerun; new weather columns go at the end."""
    return df[original_columns + [c for c in WEATHER_COLS if c not in original_columns]]

season_weather = as_text(meet_weather.loc[meet_weather['source'] == 'regular_season', ['meet_name', 'meet_date'] + WEATHER_COLS])
regular_season_out = regular_season.merge(season_weather, on=['meet_name', 'meet_date'], how='left', validate='many_to_one')
champ_weather = as_text(meet_weather.loc[meet_weather['source'] == 'championship', ['meet_year'] + WEATHER_COLS])
results_out = results.merge(champ_weather, on='meet_year', how='left', validate='many_to_one')

regular_season_out = restore_order(regular_season_out, [c for c in regular_season_columns if c in regular_season_out])
results_out = restore_order(results_out, [c for c in results_columns if c in results_out])

assert len(regular_season_out) == len(regular_season) and len(results_out) == len(results)
regular_season_out.to_csv(REGULAR_SEASON_PATH, index=False)
results_out.to_csv(RESULTS_PATH, index=False)

for name, df in [('regular_season.csv', regular_season_out), (RESULTS_PATH.name, results_out)]:
    for c in WEATHER_COLS:
        filled = df[c].ne('').sum()
        print(f'{name}: {c} filled for {filled} of {len(df)} rows ({len(df) - filled} blank)')

regular_season.csv: temp filled for 46494 of 57300 rows (10806 blank)
regular_season.csv: precip_in filled for 46494 of 57300 rows (10806 blank)
regular_season.csv: humidity_max filled for 46494 of 57300 rows (10806 blank)
bigwest_2019_2026_results.csv: temp filled for 5257 of 5257 rows (0 blank)
bigwest_2019_2026_results.csv: precip_in filled for 5257 of 5257 rows (0 blank)
bigwest_2019_2026_results.csv: humidity_max filled for 5257 of 5257 rows (0 blank)


## Check

In [8]:
display(meet_weather[meet_weather['source'] == 'championship'][['meet_name', 'meet_date', 'location'] + WEATHER_COLS])
outdoor = meet_weather[(meet_weather['source'] == 'regular_season') & ~meet_weather['is_indoor'].astype(bool)]
print('Outdoor regular-season meets missing weather:', outdoor[WEATHER_COLS].isna().any(axis=1).sum())
print('Rainiest outdoor meets:')
display(outdoor.sort_values('precip_in', ascending=False)[['meet_name', 'meet_date', 'location'] + WEATHER_COLS].head(5))
print('Share of outdoor meets with any measurable precipitation (>= 0.01 in):',
      f"{(outdoor['precip_in'] >= 0.01).mean():.0%}")

,meet_name,meet_date,location,temp,precip_in,humidity_max
437,Big West Championships 2019,"May 10-11, 2019","UC Santa Barbara (Outdoor) - Santa Barbara, CA",68.3,0.02,91.5
438,Big West Championships 2021,"May 12-15, 2021","UC Irvine - Irvine, CA",70.1,0.02,89.2
439,Big West Championships 2022,"May 13-14, 2022","UC Davis - Davis, CA",89.8,0.00,87.5
440,Big West Championships 2023,"May 12-13, 2023","Cal St. Fullerton - Fullerton, CA",75.6,0.00,98.5
441,Big West Championships 2024,"May 10-11, 2024","Cal Poly-SLO - San Luis Obispo, CA",75.2,0.00,95.5
442,Big West Championships 2025,"May 16-17, 2025","Jack Rose Track - Long Beach, CA",65.2,0.04,88.0
443,Big West Championships 2026,"May 15-16, 2026","Jack Rose Track - Long Beach, CA",68.1,0.00,89.0


Outdoor regular-season meets missing weather: 0
Rainiest outdoor meets:


,meet_name,meet_date,location,temp,precip_in,humidity_max
85,UCSB Invitational,"Mar 10, 2023","UC Santa Barbara (Outdoor) - Santa Barbara, CA",58.2,1.84,100.0
366,Bobcat Invitational 2026,"Apr 2- 4, 2026","Texas State - San Marcos, TX",80.3,0.89,98.0
101,Vince O'Boyle Track and Field Classic,"Mar 29-30, 2024","UC Irvine - Irvine, CA",59.2,0.68,98.0
178,UCSB Invite,"Apr 26, 2025","UC Santa Barbara (Outdoor) - Santa Barbara, CA",56.8,0.68,96.0
16,2019 Cal Opener,"Mar 2, 2019","California - Berkeley, CA",55.7,0.68,99.0


Share of outdoor meets with any measurable precipitation (>= 0.01 in): 32%
